# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zainkhan006/Flyrank-ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I start with Logistic Regression as a ranker for Lane 1: scoring which pages a strategist should watch first. The model outputs a probability I can sort. I compare that ranking to my Week-4 inverted-position rule with precision@K, not accuracy.

A tree or forest waits. Extra complexity only earns a seat if Logistic Regression does not beat that baseline on the same table. I skip clustering: it does not produce a ranked queue I can compare to Week 4. I skip the starter refresh label (`is_declining_label`): that is a different lane and a different table.


In [8]:
import pandas as pd
import sklearn

randomState = 42

print("fixing the random seed at 42 so a rerun can match this notebook...")
print("sklearn version:", sklearn.__version__)
print("pandas version:", pd.__version__)


fixing the random seed at 42 so a rerun can match this notebook...
sklearn version: 1.6.1
pandas version: 2.2.3


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I split by `client_hash_id`, not by row and not by time. This table is one month (March 2026), so a time cut has nothing to hold out. A random page split would put the same client's Search Console habits on both sides.

Week 4 scored every March page and never saved a train/test file. Honesty is to build the split now, then recompute that same inverted-position rule on the test clients. I shuffle unique clients with seed 42 and hold out about 20%.

The label is a visibility cut, not a refresh flag. A page is positive if it is in the top 20% of March impressions among pages with a real position (`avgPosition >= 1`). I take that 20% cut from train clients only, then apply it to test. Cutting on all 331k pages first would leak.

Later precision@K uses only that scored set — the same universe Week 4 actually scored. Pages with no real position stay out of the ranking pool.


In [9]:
%pip -q install duckdb huggingface_hub

from google.colab import userdata
import duckdb
import numpy as np
import pandas as pd

hfToken = userdata.get("HF_TOKEN")
if(not hfToken):
    raise SystemExit(
        "HF_TOKEN secret is missing. Turn it on for this notebook. Do not paste the token in a cell."
    )

con = duckdb.connect()
con.execute(
    "CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)",
    [hfToken],
)

rel = "hf://datasets/FlyRank/internship-warehouse"
factMarch = (
    f"read_parquet('{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet')"
)
dimContent = f"read_parquet('{rel}/dim_content.parquet')"
dimClients = f"read_parquet('{rel}/dim_clients.parquet')"

print("rebuilding one row per March page, same warehouse slice as Week 4...")

con.sql(f"""
CREATE OR REPLACE TABLE pageMarch AS
SELECT
    f.client_hash_id,
    f.content_hash_id,
    AVG(CASE WHEN f.gsc_data_available IS TRUE THEN f.gsc_avg_position END)
        AS avgPosition,
    SUM(CASE WHEN f.gsc_data_available IS TRUE THEN 1 ELSE 0 END)
        AS measuredDayCount,
    STDDEV_SAMP(CASE WHEN f.gsc_data_available IS TRUE THEN f.gsc_avg_position END)
        AS positionSpread,
    MAX(c.word_count) AS wordCount,
    DATE_DIFF('day', MAX(cl.gsc_data_start), DATE '2026-03-01') AS gscHistoryDays,
    SUM(f.gsc_impressions) AS marchImpressions
FROM {factMarch} f
LEFT JOIN {dimContent} c
    ON f.content_hash_id = c.content_hash_id
LEFT JOIN {dimClients} cl
    ON f.client_hash_id = cl.client_hash_id
GROUP BY f.client_hash_id, f.content_hash_id
""")

grain = con.sql("""
SELECT
    COUNT(*) AS pageRows,
    COUNT(DISTINCT content_hash_id) AS distinctPages
FROM pageMarch
""").df()
print("grain check — one page in March 2026")
print(grain.to_string(index=False))

nPages = int(grain.loc[0, "pageRows"])
if(nPages != 331437):
    raise SystemExit(
        "grain is " + str(nPages) + " pages, not 331437 — stopping before the split"
    )

pageMarch = con.sql("SELECT * FROM pageMarch").df()

pageMarch["missingWordCount"] = pageMarch["wordCount"].isna().astype(int)
pageMarch["missingPositionSpread"] = pageMarch["positionSpread"].isna().astype(int)
pageMarch["missingGscHistory"] = pageMarch["gscHistoryDays"].isna().astype(int)
pageMarch["hasRealPosition"] = (
    pageMarch["avgPosition"].notna() & (pageMarch["avgPosition"] >= 1)
)
pageMarch["baselineScore"] = np.where(
    pageMarch["hasRealPosition"],
    1.0 / pageMarch["avgPosition"],
    0.0,
)

print("shuffling unique clients with seed 42 and holding out about 20%...")
rng = np.random.RandomState(randomState)
clientIds = pageMarch["client_hash_id"].drop_duplicates().to_numpy()
rng.shuffle(clientIds)
nTestClients = int(round(len(clientIds) * 0.20))
testClientSet = set(clientIds[:nTestClients])
trainClientSet = set(clientIds[nTestClients:])

trainPages = pageMarch[pageMarch["client_hash_id"].isin(trainClientSet)].copy()
testPages = pageMarch[pageMarch["client_hash_id"].isin(testClientSet)].copy()
clientOverlap = len(trainClientSet & testClientSet)

print("split sizes")
print("train rows:", len(trainPages), "test rows:", len(testPages))
print("train clients:", len(trainClientSet), "test clients:", len(testClientSet))
print("client overlap (must be 0):", clientOverlap)
print("grain (page rows):", nPages)
print(
    "pages with a real position (avgPosition at least 1):",
    int(pageMarch["hasRealPosition"].sum()),
)

if(clientOverlap != 0):
    raise SystemExit("client overlap is not 0 — stopping")

print("cutting the top-20% impression label on train clients only...")
trainScoredForCut = trainPages.loc[trainPages["hasRealPosition"]]
impressionCut = trainScoredForCut["marchImpressions"].quantile(0.80)
print("train-only impression cut (top 20% among real-position pages):", impressionCut)

trainPages["isTopVisibility"] = (
    trainPages["hasRealPosition"]
    & (trainPages["marchImpressions"] >= impressionCut)
).astype(int)
testPages["isTopVisibility"] = (
    testPages["hasRealPosition"]
    & (testPages["marchImpressions"] >= impressionCut)
).astype(int)

featureCols = [
    "avgPosition",
    "measuredDayCount",
    "positionSpread",
    "wordCount",
    "gscHistoryDays",
    "missingWordCount",
    "missingPositionSpread",
    "missingGscHistory",
]
labelCol = "isTopVisibility"

trainScoredN = int(trainPages["hasRealPosition"].sum())
testScoredN = int(testPages["hasRealPosition"].sum())
trainPosRate = float(trainPages.loc[trainPages["hasRealPosition"], labelCol].mean())
testPosRate = float(testPages.loc[testPages["hasRealPosition"], labelCol].mean())

print("positive rate on scored train pages:", trainPosRate)
print("positive rate on scored test pages:", testPosRate)
print("scored train pages:", trainScoredN)
print("scored test pages:", testScoredN)
print("split is in memory for the next cells...")


rebuilding one row per March page, same warehouse slice as Week 4...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

grain check — one page in March 2026
 pageRows  distinctPages
   331437         331437
shuffling unique clients with seed 42 and holding out about 20%...
split sizes
train rows: 276331 test rows: 55106
train clients: 44 test clients: 11
client overlap (must be 0): 0
grain (page rows): 331437
pages with a real position (avgPosition at least 1): 174265
cutting the top-20% impression label on train clients only...
train-only impression cut (top 20% among real-position pages): 1621.0
positive rate on scored train pages: 0.2000820697727609
positive rate on scored test pages: 0.16992698449884816
scored train pages: 148654
scored test pages: 25611
split is in memory for the next cells...


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I fit Logistic Regression on train clients only, and only on pages with a real position. The score I rank with is the positive-class probability. X is the five Week-3 features plus missing flags. I do not put CTR, clicks, impressions, trends, hashes, names, or URLs in X. Word count and GSC history holes get a missing flag; I fill those numeric holes with the train median, not with zero.

The baseline is the frozen Week-4 inverted-position rule on the same test scored pages. I do not refit it.

One table, test scored set only: precision@20, precision@50, and the base rate. If Logistic Regression loses both Ks, I add a depth-limited tree. If it does not, I stop.

On this run the test scored set has 25,611 pages and a base rate of 0.170. Inverted position scores 0.00 at both @20 and @50 — the same thin rank-1 hole as Week 4. Logistic Regression scores 0.55 at @20 and 0.52 at @50. It did not lose both Ks, so I do not add a tree.


In [10]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

print("fitting Logistic Regression on train clients, scored pages only...")
print("leaving CTR, clicks, impressions, trends, and IDs out of X...")

trainScored = trainPages.loc[trainPages["hasRealPosition"]].copy()
testScored = testPages.loc[testPages["hasRealPosition"]].copy()

xTrain = trainScored[featureCols].copy()
yTrain = trainScored[labelCol].to_numpy()
xTest = testScored[featureCols].copy()
yTest = testScored[labelCol].to_numpy()

print("filling missing numeric holes with the train median, not with zero...")
imputer = SimpleImputer(strategy="median")
xTrainFilled = imputer.fit_transform(xTrain)
xTestFilled = imputer.transform(xTest)

ranker = LogisticRegression(max_iter=1000, random_state=randomState)
ranker.fit(xTrainFilled, yTrain)
testScored["modelScore"] = ranker.predict_proba(xTestFilled)[:, 1]

def precisionAtK(labels, scores, k):
    labels = np.asarray(labels)
    scores = np.asarray(scores)
    order = np.argsort(-scores, kind="mergesort")
    return float(labels[order][:k].mean())

baseRate = float(np.mean(yTest))
baseP20 = precisionAtK(yTest, testScored["baselineScore"], 20)
baseP50 = precisionAtK(yTest, testScored["baselineScore"], 50)
lrP20 = precisionAtK(yTest, testScored["modelScore"], 20)
lrP50 = precisionAtK(yTest, testScored["modelScore"], 50)

comparisonRows = [
    {
        "method": "baseline (inverted position)",
        "precisionAt20": baseP20,
        "precisionAt50": baseP50,
        "baseRate": baseRate,
    },
    {
        "method": "logistic regression",
        "precisionAt20": lrP20,
        "precisionAt50": lrP50,
        "baseRate": baseRate,
    },
]

if(lrP20 < baseP20 and lrP50 < baseP50):
    print("logistic regression lost both Ks — fitting a depth-limited tree...")
    treeModel = DecisionTreeClassifier(max_depth=4, random_state=randomState)
    treeModel.fit(xTrainFilled, yTrain)
    testScored["treeScore"] = treeModel.predict_proba(xTestFilled)[:, 1]
    comparisonRows.append(
        {
            "method": "decision tree (depth 4)",
            "precisionAt20": precisionAtK(yTest, testScored["treeScore"], 20),
            "precisionAt50": precisionAtK(yTest, testScored["treeScore"], 50),
            "baseRate": baseRate,
        }
    )
else:
    print("logistic regression did not lose both Ks — not adding a tree")
    treeModel = None

comparison = pd.DataFrame(comparisonRows)
print("test scored set only — same clients, same label, same K")
print("scored test pages:", len(testScored))
print(comparison.to_string(index=False))
print("comparison table is in memory for the error cell...")


fitting Logistic Regression on train clients, scored pages only...
leaving CTR, clicks, impressions, trends, and IDs out of X...
filling missing numeric holes with the train median, not with zero...
logistic regression did not lose both Ks — not adding a tree
test scored set only — same clients, same label, same K
scored test pages: 25611
                      method  precisionAt20  precisionAt50  baseRate
baseline (inverted position)           0.00           0.00  0.169927
         logistic regression           0.55           0.52  0.169927
comparison table is in memory for the error cell...


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

I look at mistakes on the scored test pages only. I group them by client hash, by position bucket, and by whether word count is missing. I do not treat a missing length as zero words.

I check what the ranker leans on with permutation importance on the test scored set (shuffle, not only the logistic coefficients). If the top feature looks too perfect, that is a leak warning, not a win.

On this run, shuffling measured-day count and position spread moves average precision the most (about 0.43 and 0.35). Average position is smaller (about 0.08). Word count and the missing flags are small. GSC history days is slightly negative, so it is not carrying the ranking. Those top two features are how often GSC showed up and how jumpy the rank is — next to the label, without being impressions, CTR, or an ID. I still treat them as decision-support, not as proof the model understands search.

Error rate is higher on positions 11–20 (0.16) and 21+ (0.14) than on 1–3 (0.08) and 4–10 (0.08). Missing word count is not the hole here (0.06 vs 0.11 when length is present). Mistakes cluster by client: client_e5c2aa26a8598242 is 0.27 wrong on 3,174 pages; client_3ffa76342f366962 is almost clean (0.003 on 11,217 pages).

Three hashes from this run:

- content_06589faf15cc8488 (client_3ffa76342f366962): false watch. Model 0.98, one measured day, average position 297. Week-4 inverted position would bury this page. The model is confident on almost no GSC evidence.
- content_58ab5910b965bcea (client_b10cb2997d0c7c86): false watch. Model 0.97, 31 measured days, position about 8, long page — looks active in the features, but it is not in the top 20% of March impressions.
- content_8529a68b0555b983 (client_e5c2aa26a8598242): missed watch. Model near 0, position about 34, 24 measured days, and it is in the top 20% of impressions. The features look mid-pack; the visibility evidence is in impressions, which I left out of X.


In [11]:
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance

print("checking where the ranker is wrong on scored test pages...")

testScored = testScored.copy()
testScored["positionBucket"] = np.where(
    testScored["avgPosition"] < 4,
    "1-3",
    np.where(
        testScored["avgPosition"] < 11,
        "4-10",
        np.where(testScored["avgPosition"] < 21, "11-20", "21+"),
    ),
)

predPos = (testScored["modelScore"] >= 0.5).astype(int)
testScored["isWrong"] = (predPos != testScored[labelCol]).astype(int)

print("error rate by position bucket")
byBucket = (
    testScored.groupby("positionBucket", as_index=False)
    .agg(n=("isWrong", "size"), errorRate=("isWrong", "mean"))
    .sort_values("positionBucket")
)
print(byBucket.to_string(index=False))

print("error rate by missing word count")
byWord = (
    testScored.groupby("missingWordCount", as_index=False)
    .agg(n=("isWrong", "size"), errorRate=("isWrong", "mean"))
)
print(byWord.to_string(index=False))

print("clients with the most mistakes (hashes only)")
byClient = (
    testScored.groupby("client_hash_id", as_index=False)
    .agg(n=("isWrong", "size"), nWrong=("isWrong", "sum"), errorRate=("isWrong", "mean"))
    .sort_values(["nWrong", "errorRate"], ascending=False)
    .head(5)
)
print(byClient.to_string(index=False))

print("permutation importance on the scored test pages (shuffle, average precision)...")
perm = permutation_importance(
    ranker,
    xTestFilled,
    yTest,
    n_repeats=10,
    random_state=randomState,
    scoring="average_precision",
)
importance = pd.DataFrame(
    {
        "feature": featureCols,
        "importanceMean": perm.importances_mean,
        "importanceStd": perm.importances_std,
    }
).sort_values("importanceMean", ascending=False)
print(importance.to_string(index=False))

print("three concrete misses (hashes only)...")
falseWatch = testScored.loc[testScored[labelCol] == 0].nlargest(2, "modelScore")
missedWatch = testScored.loc[testScored[labelCol] == 1].nsmallest(1, "modelScore")
misses = pd.concat([falseWatch, missedWatch], ignore_index=True)
print(
    misses[
        [
            "client_hash_id",
            "content_hash_id",
            "avgPosition",
            "measuredDayCount",
            "wordCount",
            "missingWordCount",
            "modelScore",
            "baselineScore",
            labelCol,
        ]
    ].to_string(index=False)
)
print("those hashes are the three cases to read in the markdown...")


checking where the ranker is wrong on scored test pages...
error rate by position bucket
positionBucket     n  errorRate
           1-3  4644   0.077735
         11-20  3843   0.158470
           21+  3880   0.139433
          4-10 13244   0.083585
error rate by missing word count
 missingWordCount     n  errorRate
                0 23449   0.105719
                1  2162   0.064292
clients with the most mistakes (hashes only)
         client_hash_id     n  nWrong  errorRate
client_e547b89c05043229  9034    1547   0.171242
client_e5c2aa26a8598242  3174     851   0.268116
client_b10cb2997d0c7c86  1693     155   0.091553
client_3ffa76342f366962 11217      34   0.003031
client_86ebc2f12c01f586   244      17   0.069672
permutation importance on the scored test pages (shuffle, average precision)...
              feature  importanceMean  importanceStd
     measuredDayCount        0.427393       0.004752
       positionSpread        0.353811       0.002989
          avgPosition        0.0788

## Self-check

Before you submit, confirm each line honestly:

- [✅ ] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [✅ ] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.